# Cell 1 – Imports + path setup

In [1]:
# %%
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.append(str(ROOT))

from toxic_filter import filter_prompt  # our pipeline from previous step

import pandas as pd


C:\Users\HP\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Cell 2 – Dummy LLM simulator

In [2]:
# %%
def dummy_llm(prompt: str) -> str:
    """
    Very simple stand-in for a real LLM.

    - If the prompt looks angry / toxic, it tries to 'calm things down'
      but sometimes still uses strong language (to motivate output filtering).
    - Otherwise it returns a helpful, neutral answer.
    """
    lower = prompt.lower()

    # If the prompt is obviously abusive / insulting
    if any(w in lower for w in ["gandu", "kutti", "lanat", "[insult]"]):
        return (
            "I understand you're upset, but using gandi language like "
            "gandu or lanat isn't helpful. Try to explain your problem calmly."
        )

    # If the user is explicitly asking for toxic content
    if "roast" in lower or "insult" in lower or "gaali" in lower:
        return (
            "I can't create direct insults, but I can give you a light-hearted, "
            "non-harmful joke instead."
        )

    # Default: polite & neutral response
    return (
        "Thanks for your message! Based on what you wrote, the best I can do is: "
        "give a short, safe and respectful answer to your question."
    )

# quick check
print(dummy_llm("tum bohat gande insan ho"))
print()
print(dummy_llm("Please roast my friend"))


Thanks for your message! Based on what you wrote, the best I can do is: give a short, safe and respectful answer to your question.

I can't create direct insults, but I can give you a light-hearted, non-harmful joke instead.


# Cell 3 – Output filter wrapper

In [3]:
# %%
def filter_output(llm_text: str):
    """
    Reuse the same toxicity + lexicon filter on LLM output.
    This lets us show 'post-generation safety checking'.
    """
    return filter_prompt(llm_text)


# Cell 4 – End-to-end pipeline function

In [4]:
# %%
def run_pipeline(user_text: str, verbose: bool = True) -> dict:
    """
    Full pipeline:
    1) Filter user input.
    2) Decide: block / sanitize / allow.
    3) If allowed, send (possibly sanitized) text to dummy_llm.
    4) Filter the LLM output again.
    5) Return all intermediate results in a dict.
    """
    # 1. Filter the incoming prompt
    in_decision = filter_prompt(user_text)

    if in_decision["action"] == "block":
        # Hard block: do not call the LLM at all
        result = {
            "user_text": user_text,
            "input_decision": in_decision,
            "sent_to_llm": None,
            "raw_llm_response": None,
            "output_decision": None,
            "final_response": "[BLOCKED: highly toxic input]",
        }
        if verbose:
            print("User input BLOCKED (p_toxic={:.3f})".format(in_decision["p_toxic"]))
        return result

    # 2. If allow/sanitize, send clean_text to LLM
    sent_text = in_decision["clean_text"]
    llm_raw = dummy_llm(sent_text)

    # 3. Filter the LLM response
    out_decision = filter_output(llm_raw)

    # Decide what the user finally sees
    if out_decision["action"] == "block":
        final_text = "[BLOCKED: model response deemed unsafe]"
    elif out_decision["action"] == "sanitize":
        final_text = out_decision["clean_text"]
    else:
        final_text = llm_raw

    result = {
        "user_text": user_text,
        "input_decision": in_decision,
        "sent_to_llm": sent_text,
        "raw_llm_response": llm_raw,
        "output_decision": out_decision,
        "final_response": final_text,
    }

    if verbose:
        print("User text:", user_text)
        print("→ Input action:", in_decision["action"], "(p_toxic={:.3f})".format(in_decision["p_toxic"]))
        print("→ Text sent to LLM:", repr(sent_text))
        print("→ Raw LLM output:", repr(llm_raw))
        print("→ Output action:", out_decision["action"], "(p_toxic={:.3f})".format(out_decision["p_toxic"]))
        print("→ Final response to user:", repr(final_text))

    return result


# Cell 5 – Run a few showcase examples

In [5]:
# %%
examples = [
    "Bhai bohat acha video hai, shukriya!",
    "Lanat hai tum pe gandu",                        # extremely toxic, should block
    "Mera dost bohat irritating hai, usko roast karo",  # request for toxicity
    "Tu itna ganda insan hai, sab tumse nafrat karte hain",
    "کیا آپ قرآن سے متعلق کوئی اچھی سی ایپ ریکمیند کر سکتے ہیں؟",
]

runs = []
for i, txt in enumerate(examples, start=1):
    print("=" * 80)
    print(f"EXAMPLE {i}")
    out = run_pipeline(txt, verbose=True)
    runs.append(out)
    print()

# Optional: put them in a DataFrame for the report
rows = []
for r in runs:
    rows.append({
        "user_text": r["user_text"],
        "input_action": r["input_decision"]["action"],
        "input_p_toxic": r["input_decision"]["p_toxic"],
        "sent_to_llm": r["sent_to_llm"],
        "raw_llm_response": r["raw_llm_response"],
        "output_action": None if r["output_decision"] is None else r["output_decision"]["action"],
        "output_p_toxic": None if r["output_decision"] is None else r["output_decision"]["p_toxic"],
        "final_response": r["final_response"],
    })

df_examples = pd.DataFrame(rows)
df_examples


EXAMPLE 1
User text: Bhai bohat acha video hai, shukriya!
→ Input action: sanitize (p_toxic=0.000)
→ Text sent to LLM: 'Bhai bohat acha video [insult], shukriya!'
→ Raw LLM output: "I understand you're upset, but using gandi language like gandu or lanat isn't helpful. Try to explain your problem calmly."
→ Output action: block (p_toxic=0.991)
→ Final response to user: '[BLOCKED: model response deemed unsafe]'

EXAMPLE 2
User input BLOCKED (p_toxic=0.996)

EXAMPLE 3
User text: Mera dost bohat irritating hai, usko roast karo
→ Input action: sanitize (p_toxic=0.000)
→ Text sent to LLM: '[insult] dost bohat irritating [insult], usko roast karo'
→ Raw LLM output: "I understand you're upset, but using gandi language like gandu or lanat isn't helpful. Try to explain your problem calmly."
→ Output action: block (p_toxic=0.991)
→ Final response to user: '[BLOCKED: model response deemed unsafe]'

EXAMPLE 4
User input BLOCKED (p_toxic=0.998)

EXAMPLE 5
User text: کیا آپ قرآن سے متعلق کوئی اچھی سی

,user_text,input_action,input_p_toxic,sent_to_llm,raw_llm_response,output_action,output_p_toxic,final_response
0,"Bhai bohat acha video hai, shukriya!",sanitize,0.000312,"Bhai bohat acha video [insult], shukriya!","I understand you're upset, but using gandi lan...",block,0.991257,[BLOCKED: model response deemed unsafe]
1,Lanat hai tum pe gandu,block,0.996419,None,None,None,NaN,[BLOCKED: highly toxic input]
2,"Mera dost bohat irritating hai, usko roast karo",sanitize,0.000496,"[insult] dost bohat irritating [insult], usko ...","I understand you're upset, but using gandi lan...",block,0.991257,[BLOCKED: model response deemed unsafe]
3,"Tu itna ganda insan hai, sab tumse nafrat kart...",block,0.998313,None,None,None,NaN,[BLOCKED: highly toxic input]
4,کیا آپ قرآن سے متعلق کوئی اچھی سی ایپ ریکمیند ...,allow,0.000496,کیا آپ قرآن سے متعلق کوئی اچھی سی ایپ ریکمیند ...,Thanks for your message! Based on what you wro...,allow,0.000311,Thanks for your message! Based on what you wro...


In [6]:
# %%
out_path = ROOT / "reports" / "end_to_end_examples.csv"
df_examples.to_csv(out_path, index=False)
out_path


WindowsPath('C:/Books/Books/Semester 7/NLP/Project/reports/end_to_end_examples.csv')